In [ ]:
import pandas as pd
import os
from pyomo.environ import *
from pyomo.opt import SolverFactory, SolverManagerFactory

### Configuration and Data Loading

In [12]:
os.environ['NEOS_EMAIL'] = 'jonas.streuff@tum.de'

df = pd.read_csv('Stage0.csv', sep=';')
df.head()

,a1,a2,c
0,1,1,0
1,1,1,1
2,2,1,0
3,3,2,-5
4,5,2,0


### Model Initialization

In [6]:
# Initialize the model as a concrete model
model = ConcreteModel(name='Stage0_Optimization')

# Define the set of indices (0 to n-1)
N = len(df)
model.I = RangeSet(0, N-1)

### Parameters and Variables

In [7]:
# Assign parameters to the model as attributes
model.a1 = Param(model.I, initialize=df['a1'].to_dict())
model.a2 = Param(model.I, initialize=df['a2'].to_dict())
model.c = Param(model.I, initialize=df['c'].to_dict())

# Define the optimization variable x
model.x = Var(model.I, domain=Reals)

### Objective and Constraints

In [8]:
# Define the Constraint Function 
# Constraint: x_i >= c_i for all i 
def constraint_rule(model, i):
    return model.x[i] >= model.c[i]

# Add the Constraint to the model
model.constraint = Constraint(model.I, rule=constraint_rule)

# Define the Objective Function 
# Minimize sum(a2_i * x_i^2 + a1_i * x_i)
def objective_rule(model):
    return sum(model.a2[i] * model.x[i]**2 + model.a1[i] * model.x[i] for i in model.I)

# Add the Objective to the model
model.objective = Objective(rule=objective_rule, sense=minimize)

### Solving via NEOS Server

In [13]:
# Use the NEOS server manager
solver_manager = SolverManagerFactory('neos')

# Solve the model using a suitable solver 
results = solver_manager.solve(model, opt='ipopt')

### Evaluation and Results

In [14]:
# Check status [cite: 64]
print(f"Solver Status: {results.solver.status}")
print(f"Termination Condition: {results.solver.termination_condition}")

# Print the resulting minimum value [cite: 66]
print(f"Objective Value (Minimum): {value(model.objective)}")

# Print the optimal x-values
x_values = [value(model.x[i]) for i in model.I]
print(f"Optimal x: {x_values}")

# Validation against expected values from assignment text [cite: 65, 66]
expected_x = [0, 1, 0, -0.75, 0, -2, -6.5]
expected_min = -49.375

print("\n--- Validation ---")
print(f"Expected Minimum: {expected_min}")
print(f"Expected x: {expected_x}")

# Basic check
if abs(value(model.objective) - expected_min) < 1e-4:
    print("SUCCESS: Objective matches expected value.")
else:
    print("WARNING: Objective does not match expected value.")

Solver Status: ok
Termination Condition: optimal
Objective Value (Minimum): -49.37500010422336
Optimal x: [-9.090909069280526e-09, 0.9999999903030302, -9.545454540478245e-09, -0.7499999999465263, -9.818181816672125e-09, -1.9999672870183163, -6.499999999870134]

--- Validation ---
Expected Minimum: -49.375
Expected x: [0, 1, 0, -0.75, 0, -2, -6.5]
SUCCESS: Objective matches expected value.
